# Agentic Retrieval-Augmented Generation (RAG) System

This notebook builds a document question-answering system using a RAG architecture.

Pipeline:

PDF → Chunking → Embeddings → Vector Database → Semantic Retrieval → LLM Answer

Technologies used:

- LangChain
- Sentence Transformers (all-MiniLM-L6-v2)
- Qdrant Vector Database
- Llama 3.1 via Groq
- Gradio Web Interface

# Imports

In [1]:
import os
import numpy as np
from pypdf import PdfReader

from dotenv import load_dotenv

from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.embeddings import HuggingFaceEmbeddings

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

from langchain_groq import ChatGroq

import gradio as gr


# load environment variables
load_dotenv()

# read API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Loaded:", GROQ_API_KEY is not None)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

API Loaded: True


# Extracting Text from PDFs

In [2]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
   model_name="BAAI/bge-base-en-v1.5",
)

# embeddings = embedding_model.embed_documents(
#     [chunk.page_content for chunk in chunks]
# )

# print("Number of embeddings:", len(embeddings))
# print("Embedding vector size:", len(embeddings[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(path="./qdrant_storage")
collection_name = "rag_collection"

In [4]:
# if client.collection_exists(collection_name):
#     client.delete_collection(collection_name)

# client.create_collection(
#     collection_name=collection_name,
#     vectors_config=VectorParams(
#         size=768,
#         distance=Distance.COSINE
#     )
# )

In [5]:
# points = []

# for i, embedding in enumerate(embeddings):
#     points.append(
#         PointStruct(
#             id=i,
#             vector=embedding,
#             payload={
#                 "text": chunks[i].page_content
#             }
#         )
#     )

# client.upsert(
#     collection_name=collection_name,
#     points=points
# )

# print("Embeddings stored!")

In [23]:
@tool
def search_resume(question: str):
    """
    Search the resume and return relevant sections.

    Use this tool whenever a question asks about:
    - projects
    - skills
    - education
    - experience
    - achievements
    """

    query_embedding = embedding_model.embed_query(question)

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=8
    )

    # print("\nRetrieved Chunks:\n")

    # for r in results.points:
    #     print(r.payload["text"])
    #     print("\n----------------\n")

    context = "\n".join([r.payload["text"] for r in results.points])

    return context

In [24]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key= GROQ_API_KEY,
    temperature=0
)

In [25]:
from langchain.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""
You are an intelligent assistant that analyzes documents.

You have access to the following tools:
{tools}

STRICT RULES:
- Use search_resume tool ONLY ONCE per question.
- After getting the observation, immediately give the Final Answer.
- Never call the tool more than once.

Use this format:
Question: the input question
Thought: what do I need to find?
Action: search_resume
Action Input: the search query
Observation: the result
Thought: I now know the final answer
Final Answer: answer the question clearly

Question: {input}
{agent_scratchpad}
""")

In [26]:
tools = [search_resume]

agent = create_react_agent(
    llm,
    tools,
    react_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,        # was 3, give it more room
    handle_parsing_errors=True
)

ValueError: Prompt missing required variables: {'tool_names'}

In [18]:
def ask_question(question):
    try:
        response = agent_executor.invoke(
            {"input": question}
        )

        return response.get("output", str(response))

    except Exception as e:
        return str(e)

In [19]:
def ask_llm(context, question):
    prompt = f"""
You are an AI assistant that answers questions based only on the provided context.

Context:
{context}

Question:
{question}

Answer clearly and concisely using only the context.
"""

    response = llm.invoke(prompt)

    return response.content

In [20]:
def rag_query(question):

    query_embedding = embedding_model.embed_query(question)

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=5
    )

    # print("\nRetrieved Chunks:\n")

    # for r in results.points:
    #     print(r.payload["text"])
    #     print("\n-----------------\n")

    context = "\n".join([r.payload["text"] for r in results.points])

    answer = ask_llm(context, question)

    return answer

In [21]:
# print(rag_query("What is this document about?"))

In [22]:
import gradio as gr

# Global state to track if a PDF has been indexed
indexed = False

def index_pdf(pdf_file):
    global indexed, client, embedding_model

    # Read the uploaded PDF
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()

    # Chunk it
    documents = [Document(page_content=text)]
    chunks = text_splitter.split_documents(documents)

    # Embed it
    embeddings = embedding_model.embed_documents(
        [chunk.page_content for chunk in chunks]
    )

    # Reset and re-index Qdrant
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )

    points = [
        PointStruct(
            id=i,
            vector=emb,
            payload={"text": chunks[i].page_content}
        )
        for i, emb in enumerate(embeddings)
    ]
    client.upsert(collection_name=collection_name, points=points)
    indexed = True

    return "✅ PDF indexed! You can now ask questions below."


def chat(message, history):
    if not indexed:
        return "⚠️ Please upload a PDF first."
    return ask_question(message)


with gr.Blocks(title="Agentic RAG with LLaMA 3.1") as demo:
    gr.Markdown("# 🤖 Agentic RAG with LLaMA 3.1")
    gr.Markdown("Upload any PDF, then ask questions about it.")

    with gr.Row():
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
        status = gr.Textbox(label="Status", interactive=False)

    pdf_input.change(fn=index_pdf, inputs=pdf_input, outputs=status)

    gr.ChatInterface(
        fn=chat,
        examples=[
            "What is this document about?",
            "Summarize the key points.",
            "What are the main skills mentioned?"
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.




> Entering new AgentExecutor chain...
Question: Summarize the key points.
Thought: To summarize the key points, I need to search the resume for relevant sections such as education, experience, skills, projects, and achievements.
Action: search_resume
Action Input: education experience skills projects achievementsNetworks.
Leadership & Extracurriculars
• Event Lead, JICPC (ICPC-Style Coding Contest): Spearheaded college-wide competitive programming event
managing 100+ participants, coordinating logistics, problem-setting, and platform setup.
• Technical Mentor, Divide and Conquer Coders Club: Mentored students in DSA and competitive
programming; conducted weekly coding workshops and contest preparation sessions.
programming; conducted weekly coding workshops and contest preparation sessions.
• Class Representative: Served as primary liaison between faculty and 70+ students for academic coordination and
student welfare initiatives.
(1850+)
• Solved 3000+ problems across Codeforces, Cod